The following cells acheive 2 objectives each:

1 - Runs the vision controller logic on an input video and saves an output video

2 - Prints the FPS and Average time to process each frame

Video 1

In [ ]:
import cv2
import numpy as np
import time  

class VisionControllerHUD:
    def __init__(self, width, height):
        self.w, self.h = width, height
        self.total_screen_pixels = self.w * self.h

        self.roi_points = np.array([
            [int(self.w * 0.42), int(self.h * 0.70)], 
            [int(self.w * 0.58), int(self.h * 0.70)], 
            [int(self.w * 0.75), int(self.h * 0.95)], 
            [int(self.w * 0.25), int(self.h * 0.95)]  
        ], np.int32)

    def process_frame(self, frame, frame_count):
        lower_green = np.array([0, 200, 0])
        upper_green = np.array([100, 255, 100])
        lower_red = np.array([0, 0, 200])
        upper_red = np.array([100, 100, 255])
        lower_purple = np.array([150, 0, 150])
        upper_purple = np.array([255, 100, 255])
        
        lower_cyan = np.array([180, 180, 0])   
        upper_cyan = np.array([255, 255, 120]) 

        mask_human_full = cv2.inRange(frame, lower_green, upper_green)
        mask_obstacle_full = cv2.inRange(frame, lower_red, upper_red)
        screen_hazard_pixels = np.count_nonzero(mask_human_full) + np.count_nonzero(mask_obstacle_full)
        screen_percentage = (screen_hazard_pixels / self.total_screen_pixels) * 100

        roi_mask = np.zeros((self.h, self.w), dtype=np.uint8)
        cv2.fillPoly(roi_mask, [self.roi_points], 255)
        
        mask_sb_full = cv2.inRange(frame, lower_cyan, upper_cyan)
        path_speedbreaker = cv2.bitwise_and(mask_sb_full, mask_sb_full, mask=roi_mask)
        
        path_human = cv2.bitwise_and(mask_human_full, mask_human_full, mask=roi_mask)
        path_obstacle = cv2.bitwise_and(mask_obstacle_full, mask_obstacle_full, mask=roi_mask)
        
        mask_sidewalk_full = cv2.inRange(frame, lower_purple, upper_purple)
        path_sidewalk = cv2.bitwise_and(mask_sidewalk_full, mask_sidewalk_full, mask=roi_mask)

        decision = "PROCEED"
        speed = 1.0
        active_alerts = []

        if np.count_nonzero(path_human) > 5000 or np.count_nonzero(path_obstacle) > 5000:
            decision, speed = "STOP", 0.0
            active_alerts.append("!!! HAZARD IN PATH !!!")


        elif np.count_nonzero(path_sidewalk) > 50000:
            decision, speed = "STOP", 0.0
            active_alerts.append("CRITICAL: OFF-ROAD")


        elif np.count_nonzero(path_speedbreaker) > 5000: 
            decision, speed = "SLOW", 0.4
            active_alerts.append("CAUTION: SPEEDBREAKER")


        elif screen_percentage > 30.0:
            decision, speed = "SLOW (VISIBILITY LOW)", 0.3
            active_alerts.append(f"CAUTION: {screen_percentage:.1f}% SCREEN BLOCKED")

        return decision, speed, active_alerts, screen_percentage

input_path = 'E:/Uni/Spring 25-26/AI for Robotics/Project/segmentation_output2.mp4'
output_path = 'E:/Uni/Spring 25-26/AI for Robotics/Project/vision_controller_output2.mp4'

cap = cv2.VideoCapture(input_path)
width, height = int(cap.get(3)), int(cap.get(4))
out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), cap.get(5), (width, height))
controller = VisionControllerHUD(width, height)

frame_count = 0
total_processing_time = 0  

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    frame_count += 1

    start_time = time.time()

    decision, speed, alerts, screen_pct = controller.process_frame(frame, frame_count)

    status_color = (255, 255, 255)
    if "STOP" in decision: status_color = (0, 0, 255)
    if "SLOW" in decision: status_color = (0, 255, 255)

    cv2.rectangle(frame, (15, 15), (520, 165), (0, 0, 0), -1)
    cv2.putText(frame, f"STATUS: {decision}", (30, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.9, status_color, 2)
    cv2.putText(frame, f"SPEED: {speed}x", (30, 95), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 1)
    cv2.putText(frame, f"SCREEN COVERAGE: {screen_pct:.1f}%", (30, 135), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 1)
    
    for i, alert in enumerate(alerts):
        cv2.putText(frame, alert, (30, 200 + (i*35)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    cv2.polylines(frame, [controller.roi_points], isClosed=True, color=(0, 255, 0), thickness=2)
    out.write(frame)

    end_time = time.time()
    total_processing_time += (end_time - start_time)

cap.release()
out.release()

if frame_count > 0:
    avg_time = (total_processing_time / frame_count) * 1000 
    print(f"\n--- Performance Summary ---")
    print(f"Total Frames Processed: {frame_count}")
    print(f"Average Time per Frame: {avg_time:.2f} ms")
    print(f"Effective FPS: {1000/avg_time:.2f}")
    print(f"Output saved to: {output_path}")


--- Performance Summary ---
Total Frames Processed: 2169
Average Time per Frame: 16.88 ms
Effective FPS: 59.25
Output saved to: E:/Uni/Spring 25-26/AI for Robotics/Project/vision_controller_output2.mp4


Video 2

In [ ]:
import cv2
import numpy as np
import time  

class VisionControllerHUD:
    def __init__(self, width, height):
        self.w, self.h = width, height
        self.total_screen_pixels = self.w * self.h

        self.roi_points = np.array([
            [int(self.w * 0.42), int(self.h * 0.70)], 
            [int(self.w * 0.58), int(self.h * 0.70)], 
            [int(self.w * 0.75), int(self.h * 0.95)], 
            [int(self.w * 0.25), int(self.h * 0.95)]  
        ], np.int32)

    def process_frame(self, frame, frame_count):
        lower_green = np.array([0, 200, 0])
        upper_green = np.array([100, 255, 100])
        lower_red = np.array([0, 0, 200])
        upper_red = np.array([100, 100, 255])
        lower_purple = np.array([150, 0, 150])
        upper_purple = np.array([255, 100, 255])
        
        lower_cyan = np.array([180, 180, 0])   
        upper_cyan = np.array([255, 255, 120]) 

        mask_human_full = cv2.inRange(frame, lower_green, upper_green)
        mask_obstacle_full = cv2.inRange(frame, lower_red, upper_red)
        screen_hazard_pixels = np.count_nonzero(mask_human_full) + np.count_nonzero(mask_obstacle_full)
        screen_percentage = (screen_hazard_pixels / self.total_screen_pixels) * 100

        roi_mask = np.zeros((self.h, self.w), dtype=np.uint8)
        cv2.fillPoly(roi_mask, [self.roi_points], 255)
        
        mask_sb_full = cv2.inRange(frame, lower_cyan, upper_cyan)
        path_speedbreaker = cv2.bitwise_and(mask_sb_full, mask_sb_full, mask=roi_mask)
        
        path_human = cv2.bitwise_and(mask_human_full, mask_human_full, mask=roi_mask)
        path_obstacle = cv2.bitwise_and(mask_obstacle_full, mask_obstacle_full, mask=roi_mask)
        
        mask_sidewalk_full = cv2.inRange(frame, lower_purple, upper_purple)
        path_sidewalk = cv2.bitwise_and(mask_sidewalk_full, mask_sidewalk_full, mask=roi_mask)

        decision = "PROCEED"
        speed = 1.0
        active_alerts = []

        if np.count_nonzero(path_human) > 5000 or np.count_nonzero(path_obstacle) > 5000:
            decision, speed = "STOP", 0.0
            active_alerts.append("!!! HAZARD IN PATH !!!")
        elif np.count_nonzero(path_sidewalk) > 50000:
            decision, speed = "STOP", 0.0
            active_alerts.append("CRITICAL: OFF-ROAD")
        elif np.count_nonzero(path_speedbreaker) > 5000: 
            decision, speed = "SLOW", 0.4
            active_alerts.append("CAUTION: SPEEDBREAKER")
        elif screen_percentage > 30.0:
            decision, speed = "SLOW (VISIBILITY LOW)", 0.3
            active_alerts.append(f"CAUTION: {screen_percentage:.1f}% SCREEN BLOCKED")

        return decision, speed, active_alerts, screen_percentage

input_path = 'E:/Uni/Spring 25-26/AI for Robotics/Project/segmentation_output1.mp4'
output_path = 'E:/Uni/Spring 25-26/AI for Robotics/Project/vision_controller_output1.mp4'

cap = cv2.VideoCapture(input_path)
width, height = int(cap.get(3)), int(cap.get(4))
out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), cap.get(5), (width, height))
controller = VisionControllerHUD(width, height)

frame_count = 0
total_processing_time = 0 

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    frame_count += 1

    start_time = time.time()

    decision, speed, alerts, screen_pct = controller.process_frame(frame, frame_count)

    status_color = (255, 255, 255)
    if "STOP" in decision: status_color = (0, 0, 255)
    if "SLOW" in decision: status_color = (0, 255, 255)

    cv2.rectangle(frame, (15, 15), (520, 165), (0, 0, 0), -1)
    cv2.putText(frame, f"STATUS: {decision}", (30, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.9, status_color, 2)
    cv2.putText(frame, f"SPEED: {speed}x", (30, 95), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 1)
    cv2.putText(frame, f"SCREEN COVERAGE: {screen_pct:.1f}%", (30, 135), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 1)
    
    for i, alert in enumerate(alerts):
        cv2.putText(frame, alert, (30, 200 + (i*35)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    cv2.polylines(frame, [controller.roi_points], isClosed=True, color=(0, 255, 0), thickness=2)
    out.write(frame)

    end_time = time.time()
    total_processing_time += (end_time - start_time)

cap.release()
out.release()

if frame_count > 0:
    avg_time = (total_processing_time / frame_count) * 1000 
    print(f"\n--- Performance Summary ---")
    print(f"Total Frames Processed: {frame_count}")
    print(f"Average Time per Frame: {avg_time:.2f} ms")
    print(f"Effective FPS: {1000/avg_time:.2f}")
    print(f"Output saved to: {output_path}")


--- Performance Summary ---
Total Frames Processed: 1607
Average Time per Frame: 16.88 ms
Effective FPS: 59.25
Output saved to: E:/Uni/Spring 25-26/AI for Robotics/Project/vision_controller_output1.mp4
